In [1]:
# ==========================================
# Imports
# ==========================================

import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from anova_module import ModelAnalysis

In [2]:
# ==========================================
# Data Loading & Preprocessing
# ==========================================

# 1. Download the UCI dataset (ID 26)
print("Downloading the Connect-4 dataset via UCI Repo (ID: 26)...")
connect_4 = fetch_ucirepo(id=26)

# Accessing raw data (Pandas DataFrames)
X = connect_4.data.features
y = connect_4.data.targets

# 2. Integer array encoding (n x d)
print("Encoding data into integers...")

# OrdinalEncoder for the 42 cells (categories: 'b', 'o', 'x')
encoder_features = OrdinalEncoder()
X_encoded = encoder_features.fit_transform(X).astype(np.int64)

# LabelEncoder for the target (categories: 'win', 'loss', 'draw')
encoder_target = LabelEncoder()
# Using .ravel() to transform the y DataFrame into a flat vector
y_encoded = encoder_target.fit_transform(y.values.ravel()).astype(np.int64)

print(f"Feature matrix shape (n x d): {X_encoded.shape}")
print(f"Data type: {X_encoded.dtype}")

# 3. Train / Test Split
# Using random_state=42 for reproducibility and stratify for class imbalance
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# 4. Random Forest Training
print("Training the Random Forest...")
# n_estimators=100 is a good speed/accuracy trade-off
# n_jobs=-1 uses all available CPU cores
clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

# 5. Evaluation
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nAccuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=encoder_target.classes_))

# ==========================================
# Data Inspection & Mapping
# ==========================================
print("\n--- Useful Information ---")
# To see what the numbers 0, 1, 2 in X_encoded correspond to:
for i, col_name in enumerate(X.columns[:1]): # Looking at the first column only as an example
    mapping_X = dict(zip(range(len(encoder_features.categories_[i])), encoder_features.categories_[i]))
    print(f"Cell mapping (X): {mapping_X}")

# To see what the numbers in y_encoded correspond to:
mapping_y = dict(zip(range(len(encoder_target.classes_)), encoder_target.classes_))
print(f"Target mapping (y): {mapping_y}")

Encoding data into integers...
Feature matrix shape (n x d): (67557, 42)
Data type: int64
Training the Random Forest...

Accuracy: 0.8192

Classification Report:
              precision    recall  f1-score   support

        draw       0.54      0.13      0.21      1290
        loss       0.81      0.70      0.75      3327
         win       0.83      0.96      0.89      8895

    accuracy                           0.82     13512
   macro avg       0.72      0.60      0.62     13512
weighted avg       0.80      0.82      0.79     13512


--- Useful Information ---
Cell mapping (X): {0: 'b', 1: 'o', 2: 'x'}
Target mapping (y): {0: 'draw', 1: 'loss', 2: 'win'}


In [ ]:
%%time
# ==============================================
# Functional ANOVA Decomposition (MAIN EFFECTS)
# ==============================================

def f(x): # class 2
    return(clf.predict_proba(x)[:,2]) # Proba of winning

A = ModelAnalysis(X_encoded , f , 0.124 , 1 , 1e-4) #0.124% of total dimension to have all main effects
S , Matrix = A.functional_anova() # sets and f_A(X_A)
print(A.get_R2() , A.get_L2_Error() , A.get_L2_Error_rel())

Constructing Basis Matrix: 100%|██████████| 83/83 [00:01<00:00, 78.34it/s] 


Computations complete. Results ready.
0.4491710151975612 0.07301981505256158 0.1291350369813769
CPU times: user 51.9 s, sys: 8.06 s, total: 60 s
Wall time: 9.43 s


In [6]:
%%time
# ==========================================
# Functional ANOVA Decomposition
# ==========================================

A = ModelAnalysis(X_encoded , f , 7.5 , 1 , 1e-4)
S , Matrix = A.functional_anova() # sets and f_A(X_A)
print(A.get_R2() , A.get_L2_Error() , A.get_L2_Error_rel())

Constructing Basis Matrix: 100%|██████████| 5066/5066 [34:12<00:00,  2.47it/s]


Computations complete. Results ready.
0.7032825124247271 0.03933390700813979 0.06956174200745231
CPU times: user 1h 20min 54s, sys: 39min 3s, total: 1h 59min 58s
Wall time: 37min 21s
